In [2]:
import pandas as pd

In [3]:
df=pd.read_csv("clean_data/exp_web_merged.csv")

In [4]:
df.shape

(317235, 6)

In [5]:
df.head()

,client_id,variation,visitor_id,visit_id,process_step,date_time
0,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,Test,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


##Step 1-Build Funnel Format

In [6]:
funnel = df.groupby(['variation', 'process_step'])['visit_id'].nunique()
funnel = funnel.reset_index().pivot(index='process_step', columns='variation', values='visit_id')

In [7]:
print(funnel)

variation     Control   Test
process_step                
confirm         16046  21731
start           30910  33157
step_1          23548  28285
step_2          20138  24503
step_3          18300  22186


##Order Standardized

In [8]:
step_order = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
funnel = funnel.loc[step_order]

##Step 2- Drop-off Rate(Where users leave)

In [9]:
funnel['drop_off_control'] = funnel['Control'].pct_change().abs()*100
funnel['drop_off_test'] = funnel['Test'].pct_change().abs()*100

In [10]:
funnel['drop_users_control'] = funnel['Control'].diff().abs()
funnel['drop_users_test'] = funnel['Test'].diff().abs()

In [12]:
funnel.head()

variation,Control,Test,drop_off_control,drop_off_test,drop_users_control,drop_users_test
process_step,,,,,,
start,30910,33157,NaN,NaN,NaN,NaN
step_1,23548,28285,23.817535,14.693730,7362.0,4872.0
step_2,20138,24503,14.481060,13.371045,3410.0,3782.0
step_3,18300,22186,9.127024,9.455985,1838.0,2317.0
confirm,16046,21731,12.316940,2.050843,2254.0,455.0


Key Observation-

(Start-Step1)
Control drop-off: 23.82%
Test drop-off: 14.69%
                  The Test variation reduces early abandonment by ~9 percentage points, indicating a stronger first impression and smoother onboarding experiencence.

(Step_1 → Step_2)
Control: 14.48%
Test: 13.37%
                 Redesign is slightly improved.

(Step_2 → Step_3)
Control: 9.13%
Test: 9.46%
                 Drop-off remains almost unchanged
                 
(Step_3 → Confirm)
Control: 12.32%
Test: 2.05%
                Drop-off reduced by over 10 percentage points. Indicates significantly lower friction at the conversion stage

Final Insight
                Overall, the Test variation demonstrates a clear improvement in user retention across the funnel.The major reduction in drop-off at the final step suggests that the new design has user-friendly experience.


##Step 3: Conversion Rate (How efficiently users move)

1. Step-to-step

In [13]:
funnel['conv_control'] = funnel['Control'] / funnel['Control'].shift(1)
funnel['conv_test'] = funnel['Test'] / funnel['Test'].shift(1)

In [14]:
funnel['conv_control'] = (funnel['conv_control'] * 100).round(2)
funnel['conv_test'] = (funnel['conv_test'] * 100).round(2)

In [15]:
funnel.head()

variation,Control,Test,drop_off_control,drop_off_test,drop_users_control,drop_users_test,conv_control,conv_test
process_step,,,,,,,,
start,30910,33157,NaN,NaN,NaN,NaN,NaN,NaN
step_1,23548,28285,23.817535,14.693730,7362.0,4872.0,76.18,85.31
step_2,20138,24503,14.481060,13.371045,3410.0,3782.0,85.52,86.63
step_3,18300,22186,9.127024,9.455985,1838.0,2317.0,90.87,90.54
confirm,16046,21731,12.316940,2.050843,2254.0,455.0,87.68,97.95


2.Overall conversion

In [16]:
funnel['overall_conv_control'] = funnel['Control'].div(funnel.loc['start','Control']).mul(100).round(2)
funnel['overall_conv_test'] = funnel['Test'].div(funnel.loc['start','Test']).mul(100).round(2)

In [17]:
funnel.head()

variation,Control,Test,drop_off_control,drop_off_test,drop_users_control,drop_users_test,conv_control,conv_test,overall_conv_control,overall_conv_test
process_step,,,,,,,,,,
start,30910,33157,NaN,NaN,NaN,NaN,NaN,NaN,100.00,100.00
step_1,23548,28285,23.817535,14.693730,7362.0,4872.0,76.18,85.31,76.18,85.31
step_2,20138,24503,14.481060,13.371045,3410.0,3782.0,85.52,86.63,65.15,73.90
step_3,18300,22186,9.127024,9.455985,1838.0,2317.0,90.87,90.54,59.20,66.91
confirm,16046,21731,12.316940,2.050843,2254.0,455.0,87.68,97.95,51.91,65.54


(Start → Step_1)

Control: 76.18%
Test: 85.31%
                  The new design significantly improves initial engagement and reduces early friction.
(Step_1 → Step_2)
Control: 85.52%
Test: 86.63%
                 Both designs perform similarly.

(Step_2 → Step_3)
Control: 90.87%
Test: 90.54%
                 The redesign has no meaningful impact on this stage.

(Step_3 → Confirm)
Control: 87.68%
Test: 97.95%
                 The new design dramatically improves final conversion, indicating reduced friction at the most critical stage

In [18]:
funnel['conv_diff'] = funnel['overall_conv_test'] - funnel['overall_conv_control']
funnel['drop_diff'] = funnel['drop_off_control'] - funnel['drop_off_test']

In [19]:
print(funnel)

variation     Control   Test  drop_off_control  drop_off_test  \
process_step                                                    
start           30910  33157               NaN            NaN   
step_1          23548  28285         23.817535      14.693730   
step_2          20138  24503         14.481060      13.371045   
step_3          18300  22186          9.127024       9.455985   
confirm         16046  21731         12.316940       2.050843   

variation     drop_users_control  drop_users_test  conv_control  conv_test  \
process_step                                                                 
start                        NaN              NaN           NaN        NaN   
step_1                    7362.0           4872.0         76.18      85.31   
step_2                    3410.0           3782.0         85.52      86.63   
step_3                    1838.0           2317.0         90.87      90.54   
confirm                   2254.0            455.0         87.68      97.95  

Overall analysis

The Test group significantly outperforms the Control group in overall conversion, achieving a 65.54% completion rate vs 51.91% in Control (+13.63%), indicating that the new design improves end-to-end user success.

Step 1 (Biggest Early Improvement)

Conversion:

Control: 76.18%

Test: 85.31% (+9.13%)

Drop-off:

Control: 23.82%

Test: 14.69%

The Test experience dramatically reduces early-stage drop-offs, suggesting a more intuitive or engaging entry into the process.

Step 2 (Consistent Improvement)

Conversion improvement: +8.75%

Drop-off slightly lower in Test
 
 The Test design maintains momentum, continuing to outperform Control with steady improvements in user progression.

Step 3 (Minor Difference)

Conversion:

Nearly equal (Test slightly lower by -0.33% drop difference)

Performance between Test and Control converges at this step, indicating a potential friction point that affects both experiences similarly.

Final Step (CRITICAL WIN)

Conversion:

Control: 87.68%

Test: 97.95% (+10.27%)

Drop-off:

Control: 12.32%

Test: 2.05%

The Test group shows a massive reduction in final-stage drop-offs, meaning users are far more likely to successfully complete the process once they reach the final step.

The Test experience consistently reduces drop-offs across the funnel, with the most significant improvements observed at:

Entry stage (Step 1)
Final completion stage

In [21]:
funnel.head()

variation,Control,Test,drop_off_control,drop_off_test,drop_users_control,drop_users_test,conv_control,conv_test,overall_conv_control,overall_conv_test,conv_diff,drop_diff
process_step,,,,,,,,,,,,
start,30910,33157,NaN,NaN,NaN,NaN,NaN,NaN,100.00,100.00,0.00,NaN
step_1,23548,28285,23.817535,14.693730,7362.0,4872.0,76.18,85.31,76.18,85.31,9.13,9.123805
step_2,20138,24503,14.481060,13.371045,3410.0,3782.0,85.52,86.63,65.15,73.90,8.75,1.110015
step_3,18300,22186,9.127024,9.455985,1838.0,2317.0,90.87,90.54,59.20,66.91,7.71,-0.328961
confirm,16046,21731,12.316940,2.050843,2254.0,455.0,87.68,97.95,51.91,65.54,13.63,10.266097


In [23]:
print(funnel.columns)
print(funnel.index)

Index(['Control', 'Test', 'drop_off_control', 'drop_off_test',
       'drop_users_control', 'drop_users_test', 'conv_control', 'conv_test',
       'overall_conv_control', 'overall_conv_test', 'conv_diff', 'drop_diff'],
      dtype='object', name='variation')
Index(['start', 'step_1', 'step_2', 'step_3', 'confirm'], dtype='object', name='process_step')


In [24]:
funnel = funnel.reset_index()
funnel.columns.name = None
funnel.to_csv("funnel_data.csv", index=False)

In [25]:
import pandas as pd

df1 = pd.read_csv('funnel_data.csv')



In [26]:
# Reshape to long format
long_df = pd.DataFrame({
    'process_step': list(df1['process_step']) * 2,
    'variation':    ['Control'] * len(df1) + ['Test'] * len(df1),
    'users':        list(df1['Control']) + list(df1['Test']),
    'drop_off_pct': list(df1['drop_off_control']) + list(df1['drop_off_test']),
    'drop_users':   list(df1['drop_users_control']) + list(df1['drop_users_test']),
    'step_conv':    list(df1['conv_control']) + list(df1['conv_test']),
    'overall_conv': list(df1['overall_conv_control']) + list(df1['overall_conv_test']),
    'conv_diff':    list(df1['conv_diff']) * 2,
    'drop_diff':    list(df1['drop_diff']) * 2,
})

# Set step order
step_order = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
long_df['step_order'] = long_df['process_step'].map({s: i for i, s in enumerate(step_order)})
long_df = long_df.sort_values('step_order')

long_df.to_csv('funnel_long.csv', index=False)
print(long_df)

  process_step variation  users  drop_off_pct  drop_users  step_conv  \
0        start   Control  30910           NaN         NaN        NaN   
5        start      Test  33157           NaN         NaN        NaN   
1       step_1   Control  23548     23.817535      7362.0      76.18   
6       step_1      Test  28285     14.693730      4872.0      85.31   
2       step_2   Control  20138     14.481060      3410.0      85.52   
7       step_2      Test  24503     13.371045      3782.0      86.63   
3       step_3   Control  18300      9.127024      1838.0      90.87   
8       step_3      Test  22186      9.455985      2317.0      90.54   
4      confirm   Control  16046     12.316940      2254.0      87.68   
9      confirm      Test  21731      2.050843       455.0      97.95   

   overall_conv  conv_diff  drop_diff  step_order  
0        100.00       0.00        NaN           0  
5        100.00       0.00        NaN           0  
1         76.18       9.13   9.123805           1  

In [29]:
long_df.head()

,process_step,variation,users,drop_off_pct,drop_users,step_conv,overall_conv,conv_diff,drop_diff,step_order
0,start,Control,30910,NaN,NaN,NaN,100.00,0.00,NaN,0
5,start,Test,33157,NaN,NaN,NaN,100.00,0.00,NaN,0
1,step_1,Control,23548,23.817535,7362.0,76.18,76.18,9.13,9.123805,1
6,step_1,Test,28285,14.693730,4872.0,85.31,85.31,9.13,9.123805,1
2,step_2,Control,20138,14.481060,3410.0,85.52,65.15,8.75,1.110015,2
